# Primer entrenamiento de un MLP

**Capítulo 1 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_multilayer-perceptrons/mlp-implementation.ipynb` · [Lección original](https://d2l.ai/chapter_multilayer-perceptrons/mlp-implementation.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Implementación de Perceptrones Multicapa
<a id="sec_mlp-implementation"></a>

Los perceptores multicapa (MLP) no son mucho más complejos de implementar que los modelos lineales simples. La diferencia conceptual clave es que ahora concatenamos múltiples capas.


In [ ]:
import torch
from torch import nn
from laboratorio import d2l

## Implementación desde cero
Comencemos de nuevo implementando una red así desde cero.

### Inicializando parámetros de modelo
Recordemos que Fashion-MNIST contiene 10 clases, y que cada imagen consiste en una cuadrícula $28 \times 28 = 784$ de valores de píxeles en escala de grises. Como antes ignoraremos la estructura espacial entre los píxeles por ahora, por lo que podemos pensar en esto como un conjunto de datos de clasificación con 784 características de entrada y 10 clases. Para empezar, **aplicaremos un MLP con una capa oculta y 256 unidades ocultas.** Tanto el número de capas como su ancho son ajustables (se consideran hiperparametros). Típicamente, elegimos los anchos de capa para ser divisibles por potencias mayores de 2. Esto es computacionalmente eficiente debido a la forma en que la memoria se asigna y se dirige en hardware.

Una vez más, representaremos nuestros parámetros con varios tensores. Tenga en cuenta que *para cada capa*, debemos llevar un registro de una matriz de peso y un vector de sesgo. Como siempre, asignamos memoria para los gradientes de la pérdida con respecto a estos parámetros.


En el siguiente código usamos `nn.Parameter` para registrar automáticamente un atributo de clase como parámetro que será seguido por `autograd` ([Referencia sec_autograd](https://d2l.ai/chapter_preliminaries/autograd.html#sec-autograd)).


In [ ]:
class MLPScratch(d2l.Classifier):
    def __init__(self, num_inputs, num_outputs, num_hiddens, lr, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()
        self.W1 = nn.Parameter(torch.randn(num_inputs, num_hiddens) * sigma)
        self.b1 = nn.Parameter(torch.zeros(num_hiddens))
        self.W2 = nn.Parameter(torch.randn(num_hiddens, num_outputs) * sigma)
        self.b2 = nn.Parameter(torch.zeros(num_outputs))

### Modelo
Para asegurarnos de que sabemos cómo funciona todo, **aplicaremos la activación ReLU** nosotros mismos en lugar de invocar la función `relu` incorporada directamente.


In [ ]:
def relu(X):
    a = torch.zeros_like(X)
    return torch.max(X, a)

Dado que estamos haciendo caso omiso de la estructura espacial, `reshape` cada imagen bidimensional en un vector plano de longitud `num_inputs`. Finalmente, ** implementamos nuestro modelo** con sólo unas pocas líneas de código. Ya que usamos el marco integrado autograd esto es todo lo que se necesita.


### Nota docente de Hespérides

Una red densa compone transformaciones afines y activaciones: $h=\sigma(XW_1+b_1)$ y $z=hW_2+b_2$. Sin la no linealidad, dos capas afines equivalen a una sola. Los logits $z$ aún no son probabilidades; la pérdida de clasificación integra su normalización. Antes de entrenar, comprueba que el primer eje cuenta ejemplos y el último cuenta clases. En el explorador 90 puedes observar dos coordenadas ocultas sin recurrir a una proyección dimensional.

Vínculo con los apuntes: sesión 1, «Primer entrenamiento de un MLP».


In [ ]:
@d2l.add_to_class(MLPScratch)
def forward(self, X):
    X = X.reshape((-1, self.num_inputs))
    H = relu(torch.matmul(X, self.W1) + self.b1)
    return torch.matmul(H, self.W2) + self.b2

### Entrenamiento
Afortunadamente, ** el bucle de entrenamiento para MLPs es exactamente el mismo que para la regresión softmax.** Definimos el modelo, los datos y el entrenador, luego finalmente invocamos el método `fit` sobre el modelo y los datos.


In [ ]:
model = MLPScratch(num_inputs=784, num_outputs=10, num_hiddens=256, lr=0.1)
data = d2l.FashionMNIST(batch_size=256)
trainer = d2l.Trainer(max_epochs=10)
trainer.fit(model, data)

## Implementación concisa
Como es de esperar, al confiar en las API de alto nivel, podemos implementar MLPs de forma aún más concisa.

### Modelo
En comparación con nuestra implementación concisa de implementación de regresión softmax ([Referencia sec_softmax_concise](https://d2l.ai/chapter_linear-classification/softmax-regression-concise.html#sec-softmax-concise)), la única diferencia es que añadimos *dos* capas totalmente conectadas donde previamente añadimos sólo *uno*. La primera es **la capa oculta**, la segunda es la capa de salida.


In [ ]:
class MLP(d2l.Classifier):
    def __init__(self, num_outputs, num_hiddens, lr):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(nn.Flatten(), nn.LazyLinear(num_hiddens),
                                 nn.ReLU(), nn.LazyLinear(num_outputs))

Anteriormente, definimos `forward` métodos para que los modelos transformen la entrada utilizando los parámetros del modelo. Estas operaciones son esencialmente una secuencia de procesamiento: se toma una entrada y se aplica una transformación (por ejemplo, multiplicación de la matriz con pesos seguidos de la adición de sesgo), luego se utiliza repetitivamente la salida de la transformación actual como entrada a la siguiente transformación. `forward` método se define aquí. De hecho, `MLP` hereda el `forward` método de la `Module` clase ([Referencia subsec_oo-design-models](https://d2l.ai/chapter_linear-regression/oo-design.html#subsec-oo-design-models)) simplemente invocar `self.net(X)` (`X` es entrada), que ahora se define como una secuencia de transformaciones a través de la `Sequential` clase. `Sequential` la clase abstrae el proceso hacia adelante que nos permite enfocarnos en las transformaciones. `Sequential` clase trabaja en [Referencia subsec_model-construction-sequential](https://d2l.ai/chapter_builders-guide/model-construction.html#subsec-model-construction-sequential).

### Entrenamiento
**El bucle de entrenamiento** es exactamente el mismo que cuando implementamos la regresión softmax. Esta modularidad nos permite separar las cuestiones relativas a la arquitectura del modelo de las consideraciones ortogonales.


In [ ]:
model = MLP(num_outputs=10, num_hiddens=256, lr=0.1)
trainer.fit(model, data)

## Resumen
Ahora que tenemos más práctica en el diseño de redes profundas, el paso de una sola a varias capas de redes profundas ya no plantea un desafío tan significativo. En particular, podemos reutilizar el algoritmo de entrenamiento y cargador de datos. Sin embargo, note que la implementación de MLPs desde cero es desordenada: nombrar y mantener un seguimiento de los parámetros del modelo hace difícil extender los modelos. Por ejemplo, imagine querer insertar otra capa entre las capas 42 y 43. Esto podría ser ahora la capa 42b, a menos que estemos dispuestos a realizar un cambio de nombre secuencial. Además, si implementamos la red desde cero, es mucho más difícil para el framework realizar optimizaciones de rendimiento significativas.

Sin embargo, ahora han llegado al estado del arte de finales de los años 80 cuando redes profundas totalmente conectadas eran el método de elección para el modelado de redes neuronales. Nuestro siguiente paso conceptual será considerar imágenes. Antes de hacerlo, necesitamos revisar una serie de fundamentos estadísticos y detalles sobre cómo calcular modelos de manera eficiente.

## Ejercicios
1. Cambiar el número de unidades ocultas `num_hiddens` y trazar cómo su número afecta la precisión del modelo. ¿Cuál es el mejor valor de este hiperparametro?
1. Intente agregar una capa oculta para ver cómo afecta a los resultados.
1. ¿Por qué es una mala idea insertar una capa oculta con una sola neurona?
1. ¿Cómo cambiar la tasa de aprendizaje altera sus resultados? Con todos los otros parámetros fijos, ¿qué tasa de aprendizaje le da los mejores resultados? ¿Cómo se relaciona esto con el número de épocas?
1. Optimicemos conjuntamente todos los hiperparametros, es decir, la tasa de aprendizaje, el número de épocas, el número de capas ocultas y el número de unidades ocultas por capa.
    1. ¿Cuál es el mejor resultado que puedes obtener optimizando todos ellos?
    1. ¿Por qué es mucho más difícil lidiar con múltiples hiperparametros?
    1. Describir una estrategia eficiente para optimizar sobre múltiples parámetros conjuntamente.
1. Compare la velocidad del framework y la implementación desde cero para un problema desafiante. ¿Cómo cambia con la complejidad de la red?
1. Mide la velocidad del tensor--multiplicaciones de matriz para matrices bien alineadas y desalineadas. Por ejemplo, prueba para matrices con dimensión 1024, 1025, 1026, 1028 y 1032.
    1. ¿Cómo cambia esto entre GPUs y CPUs?
    1. Determine el ancho del bus de memoria de su CPU y GPU.
1. Pruebe diferentes funciones de activación. ¿Cuál funciona mejor?
1. ¿Hay alguna diferencia entre las inicializaciones de peso de la red? ¿Importa?


[Debate del original](https://discuss.d2l.ai/t/93)
